# 03. pdf-inspector 심화 실습

목표: OCR 라우팅 정책이 비용과 지연 시간에 어떤 영향을 주는지 시뮬레이션합니다. 실제 숫자는 서비스와 문서 크기에 따라 달라지므로, 여기서는 설계 사고를 익히는 데 집중합니다.

In [ ]:
from dataclasses import dataclass


@dataclass
class PdfJob:
    name: str
    page_count: int
    pages_needing_ocr: int
    has_encoding_issues: bool = False


jobs = [
    PdfJob("annual_report", 120, 0),
    PdfJob("scanned_invoice_batch", 30, 30),
    PdfJob("mixed_contract", 45, 5),
    PdfJob("broken_text_layer", 20, 0, has_encoding_issues=True),
]


def estimate_pipeline(job: PdfJob, ocr_ms_per_page=2500, local_ms_per_page=20) -> dict:
    """전체 OCR과 하이브리드 라우팅의 처리 시간을 단순 비교합니다."""
    full_ocr_ms = job.page_count * ocr_ms_per_page

    if job.has_encoding_issues:
        # 텍스트 레이어가 있어도 깨졌다면 안전하게 전체 OCR 또는 별도 검수를 선택합니다.
        hybrid_ms = full_ocr_ms
        route = "full_ocr_due_to_encoding"
    else:
        hybrid_ms = job.page_count * local_ms_per_page + job.pages_needing_ocr * ocr_ms_per_page
        route = "hybrid_page_level_ocr" if job.pages_needing_ocr else "local_only"

    saved_ms = full_ocr_ms - hybrid_ms
    return {
        "name": job.name,
        "route": route,
        "full_ocr_sec": round(full_ocr_ms / 1000, 2),
        "hybrid_sec": round(hybrid_ms / 1000, 2),
        "saved_sec": round(saved_ms / 1000, 2),
    }


for job in jobs:
    print(estimate_pipeline(job))


In [ ]:
def choose_scan_strategy(page_count: int, accuracy_priority: str) -> str:
    """문서 크기와 정확도 요구에 따라 탐지 전략을 고릅니다.

    pdf-inspector의 실제 Rust API에는 EarlyExit, Full, Sample(n), Pages(vec) 같은 전략이 있습니다.
    이 함수는 어떤 상황에서 어떤 전략을 고를지 설명하기 위한 정책 예제입니다.
    """
    if accuracy_priority == "highest":
        return "Full"
    if page_count > 500:
        return "Sample(9)"
    if accuracy_priority == "routing_speed":
        return "EarlyExit"
    return "Full"


cases = [(40, "routing_speed"), (900, "balanced"), (120, "highest")]
for page_count, priority in cases:
    print(page_count, priority, "=>", choose_scan_strategy(page_count, priority))


심화 해석:

- `EarlyExit`는 빠른 라우팅에 유리하지만 Mixed 문서를 정밀하게 판별하려면 `Full`이 더 적합합니다.
- 매우 큰 문서는 `Sample(n)`으로 비용을 줄일 수 있지만, 샘플 밖의 스캔 페이지를 놓칠 수 있습니다.
- 운영 환경에서는 분류 결과, OCR fallback 비율, 변환 실패율, 사용자가 수정한 문서 비율을 함께 모니터링해야 합니다.